In [ ]:
!pip install wordcloud --quiet

In [ ]:
!pip install boto3 s3fs --quiet

In [ ]:
import boto3
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from wordcloud import WordCloud
import re
from collections import Counter
import os
from google.colab import userdata

# Estilo global para todas las gráficas
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.family'] = 'sans-serif'

In [ ]:
# credenciales AWS
AWS_ACCESS_KEY     = userdata.get('AWS_ACCESS_KEY')
AWS_SECRET_KEY     = userdata.get('AWS_SECRET_KEY')
AWS_REGION         = 'us-east-1'

BUCKET   = 'datos-masivos-ulacit-2026v'
PREFIX   = 'results/Unsaved/2026/04/15/'

ARCHIVOS = {
    'query1': '52e674ce-f7be-4171-be95-a75c03771ac9.csv',
    'query2': '4bac6fa0-5b03-47d3-bf5b-c90e6126866d.csv',
    'query3': '11daba6d-927f-4b18-9069-b697b0fec0e0.csv',
}

s3 = boto3.client(
    's3',
    aws_access_key_id     = AWS_ACCESS_KEY,
    aws_secret_access_key = AWS_SECRET_KEY,
    region_name           = AWS_REGION
)

dfs = {}
for nombre, archivo in ARCHIVOS.items():
    key = PREFIX + archivo
    obj = s3.get_object(Bucket=BUCKET, Key=key)
    dfs[nombre] = pd.read_csv(obj['Body'])
    print(f"{nombre}: {len(dfs[nombre]):,} filas | columnas: {list(dfs[nombre].columns)}")

df_query1 = dfs['query1']
df_query2 = dfs['query2']
df_query3 = dfs['query3']

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

bars = ax.bar(
    df_query1['hour'],
    df_query1['total_tweets'],
    color=sns.color_palette('Blues_d', len(df_query1)),
    edgecolor='white',
    linewidth=0.5
)

hora_pico = df_query1.loc[df_query1['total_tweets'].idxmax(), 'hour']
bars[df_query1['hour'].tolist().index(hora_pico)].set_color('#E05A2B')
bars[df_query1['hour'].tolist().index(hora_pico)].set_label(f'Hora pico: {hora_pico}:00 h')

ax.set_xlabel('Hora del día (UTC)', fontsize=12)
ax.set_ylabel('Cantidad de tweets', fontsize=12)
ax.set_title('Actividad de tweets por hora del día', fontsize=14, fontweight='bold', pad=15)
ax.set_xticks(df_query1['hour'])
ax.set_xticklabels([f'{h:02d}h' for h in df_query1['hour']], rotation=45, ha='right')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.legend(fontsize=11)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('grafica1_actividad_por_hora.png', dpi=150, bbox_inches='tight')
plt.show()
print("Gráfica 1 guardada.")

In [ ]:
orden_dias  = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
nombres_es  = ['Lunes','Martes','Miércoles','Jueves','Viernes','Sábado','Domingo']

resumen = df_query1.groupby('day_of_week').agg(
    total=('total_tweets', 'sum'),
    pct_pos=('pct_positive', 'mean')
).reindex(orden_dias, fill_value=0).reset_index()

positivos = resumen['total'] * resumen['pct_pos'] / 100
negativos = resumen['total'] * (1 - resumen['pct_pos'] / 100)

fig, ax = plt.subplots(figsize=(11, 5))

ax.plot(nombres_es, positivos.values, marker='o', linewidth=2.5,
        color='#2E86AB', label='Positivos', markersize=7)
ax.plot(nombres_es, negativos.values, marker='s', linewidth=2.5,
        color='#E05A2B', label='Negativos', markersize=7, linestyle='--')
ax.fill_between(nombres_es, positivos.values, negativos.values,
                alpha=0.08, color='#2E86AB')

ax.set_xlabel('Día de la semana', fontsize=12)
ax.set_ylabel('Cantidad de tweets', fontsize=12)
ax.set_title('Tweets positivos vs negativos por día de la semana',
             fontsize=14, fontweight='bold', pad=15)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.legend(fontsize=11)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('grafica2_positivos_vs_negativos.png', dpi=150, bbox_inches='tight')
plt.show()
print("Gráfica 2 guardada.")

In [ ]:
top20 = df_query3.head(20).sort_values('frecuencia', ascending=True)

fig, ax = plt.subplots(figsize=(11, 7))

bars = ax.barh(
    top20['hashtag'],
    top20['frecuencia'],
    color=sns.color_palette('viridis', len(top20)),
    edgecolor='white',
    linewidth=0.5
)

ax.set_xlabel('Frecuencia', fontsize=12)
ax.set_title('Top 20 hashtags más frecuentes', fontsize=14, fontweight='bold', pad=15)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('grafica3_top_hashtags.png', dpi=150, bbox_inches='tight')
plt.show()
print("Gráfica 3 guardada.")
